In [1]:
!pip install milvus
!pip install pymilvus

In [2]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertTokenizer, BertForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('..', 'EDP')))

In [3]:
print(sys.version)

3.11.9 (main, Apr  2 2024, 08:25:04) [Clang 15.0.0 (clang-1500.1.0.2.5)]


In [4]:
from Thesis.NIR.parser_2 import Parser2

In [5]:
xml_file_path = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/output_solr34.xml'

data = Parser2.XLMtoString(xml_file_path)

In [6]:
display(data[:10])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_426',
  'text': 'Portrait of an Officer by Jan Anthonisz van Ravesteyn, dated 1611 - 1611'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_432',
  'text': 'Portrait of the Brothers Gaspard (1519-1572), Odet (1517-1571) and François (1512-1569) de Châtillon-Coligny by Anoniem (Frankrijk), dated 1579 - 1579'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_354',
  'text': 'St Barbara by Parmigianino, dated None'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_181',
  'text': "Reinier Adriaansz's Declaration of Love by Cornelis Troost, dated 1737 - 1737"},
 {'id': 'ID: /2021672/resource_document_mauritshuis_195',
  'text': 'A Fishmo

In [14]:
type(data[0])

dict

In [7]:
# tokenizer = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
# model = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil")

# Load the tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
model = BertForMaskedLM.from_pretrained('bert-base-multilingual-cased')

Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
data[0]["text"]

'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'

In [9]:
import torch 

def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].nonzero().squeeze().detach().cpu().numpy()

In [10]:
def builder(records: list):
    ids = [x['id'] for x in records]
    text = [x['text'] for x in records]
    # create sparse vecs
    tokens = tokenizer(
        text, return_tensors='pt',
        padding=True, truncation=True
    )
    sparse_vecs = get_max_logits(model(**tokens), tokens)
    upserts = []
    for _id, sparse_vec, text in zip(ids, sparse_vecs, text):
        upserts.append({
            'id': _id,
            'sparse_values': sparse_vec,
            'text': text
        })
    return upserts

In [11]:
builder(data[:2])

[{'id': 'ID: /2021672/resource_document_mauritshuis_397',
  'sparse_values': array([  0, 100]),
  'text': 'Portrait of Helena Grondt (1613/14-after 1665) by Abraham van den Tempel, dated 1660 - 1660'},
 {'id': 'ID: /2021672/resource_document_mauritshuis_340',
  'sparse_values': array([  0, 106]),
  'text': 'The Annunciation by Francesco Solimena, dated 1693 - 1693'}]

In [12]:
from milvus import default_server

from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility, connections

In [ ]:
default_server.start()

In [ ]:
# Disconnect the existing connection if it exists
if connections.has_connection("default"):
    connections.remove_connection("default")

In [ ]:
connections.connect(
    host = '127.0.0.1', 
    port = default_server.listen_port,
    max_message_size = 64 * 1024 * 1024
)

print(utility.get_server_version())

In [ ]:
len(data)

In [ ]:
upserts = builder(data[:100])

In [ ]:
array = upserts[1]['sparse_values']

non_zeros = torch.nonzero(torch.tensor(array)).squeeze().tolist()
len(non_zeros)

In [ ]:
ids = [x['id'] for x in upserts]
sparse_embeddings = [x['sparse_values'] for x in upserts]
text = [x['text'] for x in upserts]

In [ ]:
# find max lenght of an embedding value in embeddings
max_len = max([len(x) for x in sparse_embeddings])
print(max_len)

In [ ]:
# Define the schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=50, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=max_len), 
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535)
]

schema = CollectionSchema(fields, "Schema for europeana mauritshuis data")

# Create the collection if it doesn't exist
collection_name = "eana_mauritshuis"
if not utility.has_collection(collection_name):
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection {collection_name} created.")
else:
    collection = Collection(name=collection_name)
    collection.drop()
    collection = Collection(name=collection_name, schema=schema)
    print(f"Collection {collection_name} already exists. but dropped and recreated.")


In [ ]:
print(collection.schema)
print(collection.name)

In [ ]:
milvus_data = [
    ids,
    sparse_embeddings,
    text
]
print((milvus_data[1][0]))
len(milvus_data[1][0])

In [ ]:
milvus_data_first_half = [
    ids[:len(ids)//2],
    sparse_embeddings[:len(ids)//2],
    text[:len(ids)//2]
]

milvus_data_second_half = [
    ids[len(ids)//2:],
    sparse_embeddings[len(ids)//2:],
    text[len(ids)//2:]
]

In [ ]:
collection.insert(milvus_data_first_half)

In [ ]:
collection.insert(milvus_data_second_half)

In [ ]:
index_params = {
    "metric_type": "COSINE",
    "index_type": "IVF_FLAT",
    "params": {"nlist": 1024}
}

collection.create_index(field_name="embedding", index_params=index_params)

In [ ]:
collection.load()


In [ ]:

query = input("Enter a query: ")

query_tokens = tokenizer(query, return_tensors="pt")
query_output = model(**query_tokens)

query_sparse_emb = get_max_logits(query_output, query_tokens)

search_params = {"metric_type": "COSINE", "params": {"nprobe": 10}}

results = collection.search(
    data=[query_sparse_emb],
    anns_field="embedding",
    param=search_params,
    limit=10,
    output_fields=["id", "text"],
)

print("query: ", query)
for result in results[0]:
    print(f"Document ID: {result.id}, Text: {result.entity.get('text')}, Distance: {result.distance}")

